In [2]:
import numpy as np
import os
import itertools
import logging
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import torch.nn.functional as F
from argparse import ArgumentParser
from torch.distributions import MultivariateNormal
import sys
sys.path.append("../")
import pymbar
from config import get_cfg_defaults
from nf.flows import *
from nf.models import NormalizingFlowModel
from nf.base import EinsteinCrystal
import nf.utils as util
import random
%load_ext autoreload

/home/sherryli/xsli/softwares/anaconda3/envs/sherry/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def setup_model(cfg):
    if cfg.dataset.rho is not None:
        B=(cfg.dataset.nparticles/(8*cfg.dataset.rho))**(1/3)
    else:
        B=cfg.dataset.ncellx*cfg.dataset.cell_len/2
    boxlength=2*B
    N=cfg.dataset.nparticles*3
    logging.basicConfig(level=logging.DEBUG)
    logger = logging.getLogger(__name__)  
    if cfg.prior.type=="lattice":
        prior = EinsteinCrystal(cfg.prior.lattice_dir, alpha=cfg.prior.alpha,device=cfg.device)
    elif cfg.prior.type=="normal":
        prior = MultivariateNormal(torch.zeros(N).to(cfg.device), torch.eye(N).to(cfg.device))
    if cfg.flow.type=="RealNVP":
        flows = [eval(cfg.flow.type)(dim=N,hidden_dim=cfg.flow.hidden_dim) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_AR":
        flows = [eval(cfg.flow.type)(dim=N, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_CL":
        x = [[0],[1],[2],[0,1],[1,2],[0,2]]
        mask= sum([x for _ in range(cfg.flow.nlayers//6+1)], [])[:cfg.flow.nlayers]
        flows = [eval(cfg.flow.type)(size=cfg.dataset.nparticles,dim=3, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim, mask=mask[i],device=cfg.device) for i in range(cfg.flow.nlayers)]
    model = NormalizingFlowModel(prior, flows,cfg.device).to(cfg.device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.train_parameters.learning_rate)
    #scheduler = torch. optim.lr_scheduler.ExponentialLR(optimizer, cfg.train_parameters.lr_scheduler_gamma)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.train_parameters.max_epochs)
    training_data = util.load_position(cfg.dataset.input_dir).to(cfg.device)
    with open(cfg.output.pos_dir, 'w'):
        pass
    if not(os.path.exists(cfg.output.model_dir)):
        os.mkdir(cfg.output.model_dir)
    return model,optimizer,scheduler,training_data,logger,boxlength

In [4]:
def generate_data(cfg,batchsize):
    nparticles=cfg.dataset.nparticles
    which_dist=torch.Tensor([random.randint(0,1) for _ in range(batchsize*nparticles)]).unsqueeze(-1)
    samples1=MultivariateNormal(torch.Tensor([0.5,0.5,0.5]), 0.1*torch.eye(3)).sample((batchsize*nparticles,))
    samples2=MultivariateNormal(torch.Tensor([-0.5,-0.5,-0.5]), 0.05*torch.eye(3)).sample((batchsize*nparticles,))
    return (samples1*which_dist+samples2*(1-which_dist)).reshape(batchsize,nparticles*3).to(cfg.device)

In [5]:
def train(cfg,model,optimizer,scheduler,training_data,logger,boxlength):
    with open("base.xyz", 'w'):
            pass
    losses=[]
    max_logprob=140
    lamb=0.5
    for i in range(cfg.train_parameters.max_epochs):
        optimizer.zero_grad()
        x = generate_data(cfg,cfg.train_parameters.batch_size)
        z, prior_logprob, log_det = model(x)
        util.write_coord("base.xyz",z.detach(),cfg.dataset.nparticles,boxlength)
        #x1, log_det1 =model.inverse(z)
        #print("check the map is invertible", torch.linalg.norm(x1-x), torch.linalg.norm(log_det1+log_det))
        logprob = prior_logprob + log_det
        forward_loss=-torch.mean(logprob)
        #if i>4000:
            #loss = forward_loss*lamb + (1-lamb)*reverseKL(cfg,model, cfg.train_parameters.batch_size,boxlength)
        #else:
        loss = forward_loss#-U(x,boxlength)/cfg.dataset.kT
        losses.append(loss.mean().data)
        loss.backward()
        optimizer.step()
        scheduler.step()
        if i % cfg.train_parameters.output_freq == 0:
            logger.info(f"Iter: {i}\t" +
                        f"Loss: {loss.mean().data:.2f}\t" +
                        f"Logprob: {logprob.mean().data:.2f}\t" +
                        f"Prior: {prior_logprob.mean().data:.2f}\t" +
                        f"LogDet: {log_det.mean().data:.2f}")
            samples,_,z = model.sample(1)
            util.write_coord(cfg.output.pos_dir,samples,cfg.dataset.nparticles,boxlength)
            if (i>1200) and (-forward_loss>max_logprob):
                max_logprob=-forward_loss
                torch.save({"model":model.state_dict(),"optim": optimizer.state_dict(),
                            "loss":losses},cfg.output.model_dir+cfg.dataset.name+'%d.pth'% (i//cfg.train_parameters.output_freq))

In [6]:
def read_input(dir):
    cfg = get_cfg_defaults()
    cfg.merge_from_file(dir)
    cfg.freeze()
    print(cfg)
    return cfg

In [7]:
cfg=read_input("input/mixed_gauss.yaml")
model,optimizer,scheduler,training_data,logger,boxlength = setup_model(cfg)

dataset:
  cell_len: 1
  input_dir: structures/lj.xyz
  kT: 1.0
  name: mGauss
  ncellx: 2
  ncelly: 2
  ncellz: 2
  nparticles: 20
  rho: None
device: cpu
flow:
  hidden_dim: 100
  nlayers: 6
  nsplines: 10
  type: NSF_CL
output:
  model_dir: saved_models/
  pos_dir: ./gauss_positions_during_training.xyz
prior:
  alpha: 100
  lattice_dir: structures/ref.xyz
  type: normal
train_parameters:
  batch_size: 30
  learning_rate: 0.0001
  lr_scheduler_gamma: 0.999
  max_epochs: 4000
  output_freq: 100


In [ ]:
train(cfg,model,optimizer,scheduler,training_data,logger,boxlength)

INFO:__main__:Iter: 0	Loss: 65.92	Logprob: -65.92	Prior: -65.28	LogDet: -0.64
INFO:__main__:Iter: 100	Loss: 63.91	Logprob: -63.91	Prior: -64.68	LogDet: 0.77
INFO:__main__:Iter: 200	Loss: 62.42	Logprob: -62.42	Prior: -64.87	LogDet: 2.45
INFO:__main__:Iter: 300	Loss: 59.82	Logprob: -59.82	Prior: -64.22	LogDet: 4.40
INFO:__main__:Iter: 400	Loss: 58.42	Logprob: -58.42	Prior: -63.88	LogDet: 5.46
INFO:__main__:Iter: 500	Loss: 57.54	Logprob: -57.54	Prior: -64.33	LogDet: 6.79
INFO:__main__:Iter: 600	Loss: 55.96	Logprob: -55.96	Prior: -63.97	LogDet: 8.01
INFO:__main__:Iter: 700	Loss: 55.23	Logprob: -55.23	Prior: -64.10	LogDet: 8.86
INFO:__main__:Iter: 800	Loss: 54.03	Logprob: -54.03	Prior: -64.23	LogDet: 10.20
INFO:__main__:Iter: 900	Loss: 54.06	Logprob: -54.06	Prior: -64.29	LogDet: 10.23
INFO:__main__:Iter: 1000	Loss: 53.39	Logprob: -53.39	Prior: -63.77	LogDet: 10.38
INFO:__main__:Iter: 1100	Loss: 51.95	Logprob: -51.95	Prior: -64.03	LogDet: 12.07
INFO:__main__:Iter: 1200	Loss: 52.47	Logprob: -